In [2]:
import requests
from datetime import datetime, timedelta

API_KEY = "HqrHbd6hLayqPq4rRYGJzbWFB7fFpuJNdc95Z8c2"
# Define the start date and the number of weeks to fetch (API Chaining)
print("--- NASA NEO Data Collection Tool ---")
user_start_date = input("Enter the start date (YYYY-MM-DD) [e.g., 2026-07-20]: ").strip()
total_weeks = int(input("Enter total weeks to fetch [e.g., 4]: ").strip())
try:
    current_start = datetime.strptime(user_start_date, "%Y-%m-%d")
except ValueError:
    print("❌ Invalid date format! Please use YYYY-MM-DD.")
    exit()

all_neos = []

print("Fetching data from NASA API....")
# API Chaining Loop
for i in range(total_weeks):
    current_end = current_start + timedelta(days=6)
    
    start_str = current_start.strftime("%Y-%m-%d")
    end_str = current_end.strftime("%Y-%m-%d")
    
    url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date={start_str}&end_date={end_str}&api_key={API_KEY}"
    
    try:
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            objects_by_date = data.get("near_earth_objects", {})
            
            week_count = 0
            for date_key, daily_objects in objects_by_date.items():
                all_neos.extend(daily_objects)
                week_count += len(daily_objects)
                
            print(f"Successfully fetched window from {start_str} to {end_str} ({week_count} NEOs).")
        else:
            print(f"Error fetching window {start_str}: Status code {response.status_code}")
            
    except requests.exceptions.RequestException as e:
        print(f"Network error occurred during {start_str}: {e}")
        # Move to the next week's start date
    current_start = current_end + timedelta(days=1)

print(f"\nTotal NEOs fetched and collected: {len(all_neos)}")

--- NASA NEO Data Collection Tool ---
Fetching data from NASA API....
Successfully fetched window from 2026-07-20 to 2026-07-26 (42 NEOs).
Successfully fetched window from 2026-07-27 to 2026-08-02 (29 NEOs).
Successfully fetched window from 2026-08-03 to 2026-08-09 (31 NEOs).
Successfully fetched window from 2026-08-10 to 2026-08-16 (30 NEOs).
Successfully fetched window from 2026-08-17 to 2026-08-23 (27 NEOs).

Total NEOs fetched and collected: 159


In [3]:
from pathlib import Path
import json

raw_data_dir =Path("..")/"data"/"raw"
raw_data_dir.mkdir(parents= True,exist_ok=True)

output_data_file = raw_data_dir/"all_neos_data.json"

with open(output_data_file,"w",encoding="utf_8") as f:
    json.dump(all_neos,f,indent=4)
print(f"Data successfuly saved to{output_data_file}")   

Data successfuly saved to..\data\raw\all_neos_data.json


In [ ]:
import requests
from datetime import datetime, timedelta

API_KEY = "HqrHbd6hLayqPq4rRYGJzbWFB7fFpuJNdc95Z8c2"
# Define the start date and the number of weeks to fetch (API Chaining)
start_date_str = "2026-07-20"
total_weeks = 4

current_start = datetime.strptime(start_date_str, "%Y-%m-%d")
all_neos = []

print("Fetching data from NASA API....")
# API Chaining Loop
for i in range(total_weeks):
    current_end = current_start + timedelta(days=6)
    
    start_str = current_start.strftime("%Y-%m-%d")
    end_str = current_end.strftime("%Y-%m-%d")
    
    url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date={start_str}&end_date={end_str}&api_key={API_KEY}"
    
    try:
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            objects_by_date = data.get("near_earth_objects", {})
            
            week_count = 0
            for date_key, daily_objects in objects_by_date.items():
                all_neos.extend(daily_objects)
                week_count += len(daily_objects)
                
            print(f"Successfully fetched window from {start_str} to {end_str} ({week_count} NEOs).")
        else:
            print(f"Error fetching window {start_str}: Status code {response.status_code}")
            
    except requests.exceptions.RequestException as e:
        print(f"Network error occurred during {start_str}: {e}")
        # Move to the next week's start date
    current_start = current_end + timedelta(days=1)

print(f"\nTotal NEOs fetched and collected: {len(all_neos)}")

In [ ]:
import requests

API_KEY = "HqrHbd6hLayqPq4rRYGJzbWFB7fFpuJNdc95Z8c2"
url = f"https://api.nasa.gov/neo/rest/v1/feed?start_date=2026-08-14&end_date=2026-08-20&api_key={API_KEY}"

try:
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        print("تم الاتصال بنجاح! 🎉")
        print(f"عدد الكويكبات اللي تم رصدها في الأسبوع ده: {data['element_count']}")
    else:
        print(f"حصلت مشكلة. كود الخطأ: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"مشكلة في الاتصال: {e}")

In [4]:
import json
from pathlib import Path

data_file=Path("../data/raw/all_neos_data.json")
with open(data_file,"r") as f:
    data = json.load(f)
print(len(data))

159


In [10]:
max_diameter=0
largest_asteroid_name=""

max_speed=0
fastest_asteroid_name=""

for asteroid in data:
    name=asteroid.get("name","unknown")
    diameter_max=asteroid.get("estimated_diameter",{}).get("kilometers",{}).get("estimated_diameter_max",0)

    if diameter_max>max_diameter:
        max_diameter=diameter_max
        largest_asteroid_name=name

    close_approch = asteroid.get("close_approach_data",[])
    if close_approch:
        speed_str=close_approch[0].get("relative_velocity",{}).get("kilometers_per_second",0)
        speed=float(speed_str)

        if speed>max_speed:
            max_speed=speed
            fastest_asteroid_name=name   

            
print("--- Manual EDA Results ---")
print(f"🪐 Largest Asteroid: {largest_asteroid_name} (Diameter: {max_diameter:.2f} km)")
print(f"⚡ Fastest Asteroid: {fastest_asteroid_name} (Speed: {max_speed:.2f} km/s)")
             

--- Manual EDA Results ---
🪐 Largest Asteroid: 1620 Geographos (1951 RA) (Diameter: 5.25 km)
⚡ Fastest Asteroid: 417874 (2007 NC5) (Speed: 40.01 km/s)


In [19]:
import json
from pathlib import Path

# 1. قراءة ملف الـ JSON الأساسي
json_file = Path("../data/raw/all_neos_data.json")
if not json_file.exists():
    json_file = Path("data/raw/all_neos_data.json")

with open(json_file, "r", encoding="utf-8") as f:
    neos_list = json.load(f)

# 2. استخراج الـ IDs الفريدة
neo_ids = [str(obj["neo_reference_id"]) for obj in neos_list]
neo_ids = list(dict.fromkeys(neo_ids))  # إزالة التكرار مع الحفاظ على الترتيب

# 3. حفظ الـ IDs في ملف نصي extracted_ids.txt فقط
raw_dir = json_file.parent
raw_dir.mkdir(parents=True, exist_ok=True)

ids_file = raw_dir / "extracted_ids.txt"
ids_file.write_text("\n".join(neo_ids), encoding="utf-8")

print(f"Extracted {len(neo_ids)} unique NEO ids and saved only to {ids_file}")

Extracted 159 unique NEO ids and saved only to ..\data\raw\extracted_ids.txt


In [ ]:


def safe_float(val,default=0.0):
    if val is None:
        return default
    try:
        cleaned=str(val).split()
        if not cleaned or cleaned.lower() in ["n/a","null","none"]:
            return default
        return float(val)
    except:
        return default